# ANEXO K — Síntesis multievento reproducible

Consolida exclusivamente resultados ya congelados de Valparaíso 2017, Acarí/Lomas 2018, Vallenar 2020 y Yauca/Caravelí 2024. No recalibra V5.1.

In [ ]:
import os, pandas as pd, numpy as np
BASE_DRIVE='/content/drive/MyDrive/Tesis/Tesis_Vigente/Anexos/K_Analisis_Multievento'
BASE = BASE_DRIVE if os.path.exists(BASE_DRIVE) else '/mnt/data'
FILES={
'metricas':'Comparacion_Multievento_V5_1_metricas_y_spearman.csv',
'holm':'Comparacion_Multievento_V5_1_Holm.csv',
'bias':'Heterogeneidad_BIAS_eventos.csv',
'welch':'Heterogeneidad_BIAS_Welch_ANOVA.csv',
'cochran':'Heterogeneidad_BIAS_CochranQ.csv',
'pares':'Heterogeneidad_BIAS_pares_Welch_Holm.csv',
'barrido':'Barrido_Elegibilidad_Post2016_V5_1.csv'}
D={k:pd.read_csv(os.path.join(BASE,v)) for k,v in FILES.items()}
for k,d in D.items(): print(k,d.shape)

## 1. Controles básicos de los cuatro eventos

In [ ]:
m=D['metricas'].copy()
assert len(m)==4
assert int(m['N'].sum())==116
assert set(m['Evento'])=={'Valparaiso 2017','Acari/Lomas 2018','Vallenar 2020','Yauca/Caraveli 2024'}
assert m['Spearman_rho'].between(0.646,0.904).all()
print(m[['Evento','Mw','N','R2_lnSa','RMSE_lnSa','MAE_lnSa','BIAS_pred_minus_obs','Spearman_rho','Perm_p_one_sided']])

## 2. Recalcular corrección de Holm desde p de permutación

In [ ]:
p=m[['Evento','Perm_p_one_sided']].sort_values('Perm_p_one_sided').reset_index(drop=True)
M=len(p); raw=p['Perm_p_one_sided'].to_numpy(); adj=np.empty(M); running=0
for i,val in enumerate(raw):
    candidate=(M-i)*val
    running=max(running,candidate)
    adj[i]=min(running,1.0)
p['Holm_rank']=np.arange(1,M+1); p['Holm_p_recalculado']=adj
h=D['holm'][['Evento','Holm_p_adjusted']].merge(p,on='Evento')
assert np.allclose(h['Holm_p_adjusted'],h['Holm_p_recalculado'],rtol=0,atol=1e-12)
assert (h['Holm_p_adjusted']<0.05).all()
print(h)

## 3. Heterogeneidad del BIAS — resultados diagnósticos preservados

In [ ]:
b=D['bias']; w=D['welch'].iloc[0]; q=D['cochran'].iloc[0]
assert len(b)==4
assert abs(float(w['F'])-22.782346)<1e-5
assert float(w['p_value'])<1e-7
assert abs(float(q['I2_percent'])-95.768091)<1e-5
print(b[['Evento','N','BIAS','RMSE','SD_RESIDUAL','factor_exp_bias']])
print('Welch F=',w['F'],'p=',w['p_value'])
print('Cochran Q=',q['Q'],'I2=',q['I2_percent'])

## 4. Barrido de elegibilidad previo al desempeño

In [ ]:
e=D['barrido']; elig=e[e['Estado_final'].eq('ELEGIBLE_Y_VALIDADO')]
assert set(elig['Evento'])=={'Valparaiso 2017','Acari/Lomas 2018','Vallenar 2020','Yauca/Caraveli 2024'}
print(e[['Evento','Fecha_UTC','Mw','Gate_temporal_post2016','Gate_Nazca_Sudamerica_interfaz','Gate_ruptura_finita','Gate_estaciones_ge10','Estado_final','Razon_principal']])

## 5. Auditoría final

In [ ]:
audit=pd.DataFrame([
['Eventos externos',len(m),4],['Registros externos',int(m.N.sum()),116],
['Spearman mínimo',m.Spearman_rho.min(),0.646458],['Spearman máximo',m.Spearman_rho.max(),0.903030],
['Holm significativos',(D['holm'].Holm_p_adjusted<.05).sum(),4],
['I2 (%)',float(q.I2_percent),95.768091]],columns=['Control','Obtenido','Referencia'])
print(audit)
audit.to_csv(os.path.join(BASE,'K_AUDITORIA_FINAL_MULTIEVENTO.csv'),index=False)
print('AUDITORÍA K: OK')